In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt


### Load Data from the data_frames dir
This folder has preprocessed data that will be used for next tasks

In [2]:

IN = Path.cwd() / "data_frames"

train_df     = pd.read_parquet(IN / "train_df.parquet")
val_df       = pd.read_parquet(IN / "val_df.parquet")
train_df_nl  = pd.read_parquet(IN / "train_df_nl.parquet")
val_df_nl    = pd.read_parquet(IN / "val_df_nl.parquet")

# Arrays + feature names
arr = np.load(IN / "meta_arrays_v1.npz", allow_pickle=True)
Xtr_meta, Xva_meta = arr["Xtr"], arr["Xva"]
ytr, yva = arr["ytr"], arr["yva"]
kept_feature_cols = arr["feat_cols"].tolist()

### Dataframe Summary

In [8]:
print("train_df shape:", train_df.shape)
print("val_df shape:", val_df.shape)
print("train_df_nl shape:", train_df_nl.shape )
print("val_df_nl shape:", val_df_nl.shape)
print("Xtr_meta shape:", Xtr_meta.shape)
print("Xva_meta shape:", Xva_meta.shape)
print("Length of ytr:", len(ytr))
print("Length of yva:", len(yva))
print("Kept features: ", len(kept_feature_cols))
kept_feature_cols

train_df shape: (46842, 32)
val_df shape: (9412, 32)
train_df_nl shape: (46842, 22)
val_df_nl shape: (9412, 22)
Xtr_meta shape: (46842, 18)
Xva_meta shape: (9412, 18)
Length of ytr: 46842
Length of yva: 9412
Kept features:  18


['year',
 'month',
 'day',
 'habitat',
 'countryCode',
 'hasCoordinate',
 'iucnRedListCategory',
 'substrate',
 'latitude',
 'longitude',
 'coorUncert',
 'region',
 'district',
 'metaSubstrate',
 'poisonous',
 'elevation',
 'landcover',
 'biogeographicalRegion']

In [ ]:
# data types of the columns
train_df.dtypes

eventDate                 object
year                       int64
month                    float64
day                      float64
habitat                   object
countryCode               object
scientificName            object
kingdom                   object
phylum                    object
class                     object
order                     object
family                    object
genus                     object
specificEpithet           object
hasCoordinate               bool
species                   object
iucnRedListCategory       object
substrate                 object
latitude                 float64
longitude                float64
coorUncert               float64
observationID              int64
region                    object
district                  object
filename                  object
category_id                int64
metaSubstrate             object
poisonous                  int64
elevation                float64
landcover                float64
biogeograp

In [10]:
# how many rows per species (descending)
print("Species Frequency in train set")
counts_train = train_df['species'].value_counts()
print(counts_train) 
print("\nSpecies Frequency in val set")
counts_val = val_df['species'].value_counts()
print(counts_val) 

Species Frequency in train set
species
Clitocybe nebularis     1811
Mycena galericulata     1643
Boletus edulis          1397
Amanita muscaria        1376
Amanita rubescens       1046
                        ... 
Mycena picta              13
Amanita simulans          13
Russula albonigra         10
Clitocybe barbularum       9
Mycena scirpicola          8
Name: count, Length: 210, dtype: int64

Species Frequency in val set
species
Boletus edulis             621
Amanita muscaria           517
Clitocybe nebularis        337
Russula adusta             261
Mycena rosea               254
                          ... 
Mycena citrinomarginata      1
Mycena capillaripes          1
Mycena pseudopicta           1
Russula aquosa               1
Mycena xantholeuca           1
Name: count, Length: 189, dtype: int64


In [12]:
print("Overall check for missing data:")
for name, df in [("train", train_df), ("val", val_df)]:
    print(name, df.shape)
    print("missing %:\n", (df.isna().mean()*100).round(1).sort_values(ascending=False).head(10))
    print("sample cols:", df.columns.tolist()[:12], "\n")


Overall check for missing data:
train (46842, 32)
missing %:
 substrate                3.2
coorUncert               1.5
district                 0.3
region                   0.3
biogeographicalRegion    0.2
landcover                0.1
elevation                0.1
eventDate                0.0
year                     0.0
poisonous                0.0
dtype: float64
sample cols: ['eventDate', 'year', 'month', 'day', 'habitat', 'countryCode', 'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family'] 

val (9412, 32)
missing %:
 biogeographicalRegion    0.1
district                 0.1
region                   0.1
eventDate                0.0
year                     0.0
landcover                0.0
elevation                0.0
poisonous                0.0
metaSubstrate            0.0
category_id              0.0
dtype: float64
sample cols: ['eventDate', 'year', 'month', 'day', 'habitat', 'countryCode', 'scientificName', 'kingdom', 'phylum', 'class', 'order', 'family'] 



In [16]:
# how many unique species in train_df
n_species_train = train_df['species'].nunique()
print("unique species:", n_species_train)
# how many unique species in val_df
n_species_val = val_df['species'].nunique()
print("unique species:", n_species_val)

unique species: 210
unique species: 189
